# Notebook 04 - End-to-End：MCP 工具层 + 服务编排 + 持久化 + 滚动调度

这是进阶 Notebook：把你关心的 10 层真正串成业务闭环，回答“最终是具体做什么事情”。

核心结论先说：
- 这套库是一个**可通过 MCP 暴露能力的水库调度决策内核**
- 它支持从“当前状态 + 预报”出发，生成候选方案、仿真、评估、解释，并落库留痕
- 在滚动调度场景下还能持续重评估、替换 working plan、最终定版（finalize）


## 1) 先看 MCP 工具层暴露了什么能力

你可以把 `tools/*.py` 理解成“给外部（Agent/系统）调用的函数门面”。

在 `pyresops.server` 中，这些工具会通过 `FastMCP` 注册。


In [ ]:
from pathlib import Path

tool_files = sorted(Path('pyresops/tools').glob('*_tools.py'))
print('MCP tool files:')
for p in tool_files:
    print('-', p.name)


## 2) 用服务层模拟一次完整业务流程（不依赖 MCP 客户端）

这里我们直接调用 `services`，本质和 MCP 工具调用后最终进入的逻辑是同一套。


In [ ]:
from datetime import datetime, timedelta

from pyresops.domain.reservoir import ReservoirSpec, LevelStorageCurve, DischargeCapacity
from pyresops.domain.forecast import ForecastBundle, ForecastSeries
from pyresops.domain.program import TimeHorizon
from pyresops.services import SnapshotService, ProgramService, SimulationService, EvaluationService, ExplanationService

spec = ReservoirSpec(
    id='nb4_res',
    name='Notebook 4 Reservoir',
    dead_level=150.0,
    normal_level=175.0,
    flood_limit_level=155.0,
    design_flood_level=178.0,
    check_flood_level=182.0,
    total_capacity=39.3,
    flood_capacity=22.15,
    level_storage_curve=LevelStorageCurve(levels=[135,145,155,165,175,185], storages=[0,10,20,30,39.3,51.6]),
    discharge_capacity=DischargeCapacity(levels=[135,145,155,165,175,185], max_discharges=[0,5000,10000,15000,20000,30000]),
)

snapshot_service = SnapshotService()
program_service = ProgramService()
simulation_service = SimulationService(spec, program_service.get_module_registry())
evaluation_service = EvaluationService(spec)
explanation_service = ExplanationService()

state0 = snapshot_service.create_initial_snapshot('nb4_res', spec, level=160.0, inflow=7000.0)

start = datetime(2024, 7, 3, 0, 0, 0)
forecast = ForecastBundle(
    forecast_time=start,
    series=[
        ForecastSeries(
            variable='inflow',
            timestamps=[start + timedelta(hours=i) for i in range(24)],
            values=[7000 + (i * 500 if i < 10 else 5000 - (i-10)*300) for i in range(24)],
        )
    ],
)

program = program_service.create_program(
    name='nb4 pipeline program',
    time_horizon=TimeHorizon(start=start, end=start + timedelta(hours=23), time_step=3600),
    module_configs=[{'module_type': 'constant_release', 'parameters': {'target_flow': 8000.0}}],
)

sim_result = simulation_service.run_simulation(program, state0, forecast)
eval_result = evaluation_service.evaluate(sim_result, include_step_scores=True)
explain = explanation_service.explain_program(program, sim_result, eval_result)

print('program_id =', program.id)
print('simulation max_level =', round(sim_result.max_level, 3))
print('evaluation overall_score =', round(eval_result.overall_score, 3))
print('explanation keys =', list(explain.keys()))


## 3) 看持久化层：为什么需要 `storage.Repository`

如果没有 `storage`，每次推演都只是临时结果。

`Repository` 让你可以：
- 保存方案、仿真、评估
- 写事件日志（例如人工干预）
- 保存 decision trace，支持复盘
- 保存 finalized 历史版本


In [ ]:
from pyresops.storage import Repository

repo = Repository(':memory:')

repo.save_program(program.id, {
    'program_id': program.id,
    'name': program.name,
    'created_at': program.created_at.isoformat(),
    'modules': [m.model_dump() for m in program.module_sequence],
})

repo.save_simulation_result(program.id, {
    'program_id': program.id,
    'start_time': sim_result.start_time.isoformat(),
    'end_time': sim_result.end_time.isoformat(),
    'max_level': sim_result.max_level,
    'min_level': sim_result.min_level,
    'avg_outflow': sim_result.avg_outflow,
    'snapshot_count': len(sim_result.snapshots),
})

repo.save_evaluation_result(program.id, {
    'overall_score': eval_result.overall_score,
    'flood_control_score': eval_result.flood_control_score,
    'water_supply_score': eval_result.water_supply_score,
    'violations_count': len(eval_result.constraint_violations),
})

repo.log_event(
    event_type='manual_override',
    reservoir_id='nb4_res',
    program_id=program.id,
    description='operator adjusted outflow in first 2 hours',
    data={'hours': [0,1], 'delta': 1000},
)

events = repo.list_events()
print('saved programs:', len(repo.list_programs()))
print('events:', len(events))
print('latest event:', events[0]['event_type'], '|', events[0]['description'])

repo.close()


## 4) 滚动调度闭环：`RollingOpsService` 为什么关键

在真实调度中，预报会不断更新，所以不是“一次决策就结束”。

`RollingOpsService` 提供了工作流能力：
- `optimize_flexible_release_plan`: 生成候选方案 + 证据
- `reassess_plan`: 外部条件更新后重评估
- `replace_working_plan`: 显式替换当前 working plan
- `finalize_plan`: 定版并持久化历史
- `get_working_state`: 查询当前工作态


In [ ]:
from pyresops.services import OptimizationService, RollingOpsService
from pyresops.storage import Repository

repo2 = Repository(':memory:')
opt_service = OptimizationService(spec, program_service)
rolling = RollingOpsService(
    program_service=program_service,
    simulation_service=simulation_service,
    evaluation_service=evaluation_service,
    optimization_service=opt_service,
    snapshot_service=snapshot_service,
    repository=repo2,
)

forecast_roll = ForecastBundle(
    forecast_time=start,
    series=[
        ForecastSeries(
            variable='inflow',
            timestamps=[state0.timestamp + timedelta(hours=i) for i in range(24)],
            values=[8000 + (i * 300 if i < 8 else max(0, 2400 - (i-8)*200)) for i in range(24)],
        )
    ],
)

candidate = rolling.optimize_flexible_release_plan(
    reservoir_id='nb4_res',
    context_id='ctx-001',
    horizon_hours=12,
    control_interval_seconds=3600,
    forecast=forecast_roll,
    constraints={'max_outflow': 18000.0, 'min_environmental_flow': 3000.0},
    objectives={'flood': 0.6, 'supply': 0.4},
    directives={'safety_factor': 0.9},
)

state = rolling.get_working_state(reservoir_id='nb4_res', context_id='ctx-001')

print('candidate_plan_id:', candidate['candidate_plan_id'])
print('auto_adopted:', candidate['summary']['auto_adopted_as_working'])
print('working_plan_id:', state['working_plan_id'])
print('candidate ids:', state['candidate_plan_ids'])

repo2.close()


## 5) 回到你的问题：最终串到一起具体在做什么？

一句话：

> 这个项目把“水库调度经验”工程化成一套可被人类和 MCP 客户端共同调用的决策系统。

更具体是：
1. **输入层**：快照 + 预报 + 策略（规则/约束/目标）
2. **计算层**：仿真引擎按时间步推进，给出可执行出流
3. **治理层**：规则与约束动态纠偏，保证可控与合规
4. **评价层**：多指标评分，支持方案比较
5. **服务层**：把上述能力封装成稳定 API/工具
6. **持久化层**：可追溯、可复盘、可审计
7. **滚动层**：应对条件变化，持续更新 working plan

所以它不是“单次算一个数字”，而是“持续决策系统”。


## 6) 建议学习顺序

- 先跑 `examples/case1_flood_dispatch.py`
- 再跑 `examples/case2_program_comparison.py`
- 然后看 `examples/case3_persistence.py`
- 最后阅读 `pyresops/services/rolling_ops.py` 的 5 个核心方法

到这一步，你就可以把项目当“可扩展的水库调度操作系统”来理解。
